# Which lever moved?

**Kalpa Retail, Week 1 Day 2.** Meera has read yesterday's tree and replied:

> "So revenue is customers, times how often they buy, times basket, times price. Now: which of
> those moved? Q2 was Rs 1.9 crore, Q1 was 2.1. Are we losing customers, or are the ones we have
> buying less? Marketing says more customers. Prove it or disprove it."

A second message arrived at the same time. The head of Retail-Plus forwards a member's complaint
that **the app's reorder button has not worked for six weeks**, and asks whether his tier is the
one slipping.

This notebook climbs the investigation ladder in order. It does not skip to the answer, because
the order is the skill.

In [1]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

ORDERS = kit.load_records("C2_W01_D02_orders_STUDENT.py")
print(f"{len(ORDERS)} rows across two quarters")
print(ORDERS[0])

200 rows across two quarters
{'order_id': 'KR-02001', 'customer_id': 'C-2000', 'segment': 'Retail-Core', 'channel': 'web', 'city': 'Mumbai', 'order_date': '2026-04-22', 'amount': 2200, 'status': 'delivered', 'quarter': 'Q1', 'discount': 50}


## MAP: five rungs, and you never skip one

Most wrong answers to this case are somebody starting at rung five because they arrived with a
theory. The complaint about the reorder button is exactly such a theory, and it was handed to you
by the person whose tier it would excuse.

In [2]:
kit.ladder(["confirm the drop is real",
            "compare like with like",
            "decompose along the tree",
            "isolate the branch and the segment",
            "hypothesise, and say what settles it"],
           lit=0, title="The sales-drop investigation ladder")

## Rung 1: confirm the drop is real

Counting per group is yesterday's accumulator kept per key rather than once.

In [3]:
revenue = {}
orders = {}
for order in ORDERS:
    q = order["quarter"]
    revenue[q] = revenue.get(q, 0) + order["amount"]
    orders[q] = orders.get(q, 0) + 1

kit.table(["quarter", "rows", "revenue"],
          [(q, orders[q], f"Rs {revenue[q]:,}") for q in sorted(revenue)],
          caption="Rung 1, before anything is explained")
drop = 100 * (revenue["Q2"] / revenue["Q1"] - 1)
print(f"revenue change: {drop:+.1f} percent")

quarter,rows,revenue
Q1,114,"Rs 21,000,000"
Q2,86,"Rs 18,700,000"


revenue change: -11.0 percent


The drop is real and it is 11 percent. Meera said 2.1 against 1.9, and the file agrees with her,
which is itself worth saying out loud: the first thing rung one can find is that the stakeholder's
own number is wrong.

## The break: a field that is not always there

The tree has a discounts branch, so the obvious next move is to total the discounts.

In [4]:
try:
    discounts = 0
    for order in ORDERS:
        discounts = discounts + order["discount"]
    print("discounts:", discounts)
except KeyError as e:
    print("KeyError:", e)

KeyError: 'discount'


### Reading it

`KeyError: 'discount'` says the key is not in that record. Python refuses to guess, because a
silent zero would make a missing discount look like a discount of nothing, and those are different
facts about the business.

So the first question is not how to suppress it. It is **how many records is this, and are they a
random subset or a meaningful one?**

In [5]:
absent = [o for o in ORDERS if "discount" not in o]
by_segment = {}
for o in absent:
    by_segment[o["segment"]] = by_segment.get(o["segment"], 0) + 1

print(f"records with no discount field: {len(absent)} of {len(ORDERS)}")
kit.table(["segment", "records with no discount field"],
          sorted(by_segment.items()), caption="Is the absence concentrated anywhere?")

records with no discount field: 58 of 200


segment,records with no discount field
Business,10
Retail-Core,18
Retail-Plus,18
Student,12


The fix is `.get()` **with a default you chose on purpose**, and the reason belongs in the code
rather than in somebody's memory.

In [6]:
# Default of 0: a record with no discount field is a record where no discount was applied, which is
# what the source system means by omitting it. If it turned out to mean "not captured", this line
# would be wrong and the number below would be an understatement. Flagged for Wednesday.
discounts = 0
for order in ORDERS:
    discounts = discounts + order.get("discount", 0)
print(f"discounts: Rs {discounts:,} across {len(ORDERS)} rows")

discounts: Rs 9,600 across 200 rows


## Rung 2: compare like with like

Before any factor is blamed, the two quarters have to be the same kind of thing. The one that
matters most here is the customer base, because marketing's whole claim rests on it.

In [7]:
customers = {}
for order in ORDERS:
    customers.setdefault(order["quarter"], set()).add(order["customer_id"])

kit.table(["quarter", "distinct customers"],
          [(q, len(customers[q])) for q in sorted(customers)],
          caption="Rung 2, the base marketing says is shrinking")

quarter,distinct customers
Q1,69
Q2,69


In [8]:
kit.check("the drop is real and about 11 percent", -12 < drop < -10, f"{drop:+.1f} percent")
kit.check("the customer count is flat", len(customers["Q1"]) == len(customers["Q2"]),
          f"{len(customers['Q1'])} then {len(customers['Q2'])}")
kit.check("the discount field is absent on a real subset", 40 < len(absent) < 80,
          f"{len(absent)} records")

**Marketing's claim is disproved.** Sixty-nine distinct customers bought in each quarter. The same
people are still here; they are coming back less often, which is a different problem with a
different bill attached.

## Write it once, because you need it twenty times

Four numbers, per quarter, then per segment inside each quarter. A function is one decision
applied everywhere, and it returns rather than prints so the next line can build on it.

In [9]:
def to_amount(value, default=0):
    """int(), except that a bad value costs one row rather than the whole loop."""
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def describe(rows):
    people = {o["customer_id"] for o in rows}
    total = sum(to_amount(o["amount"]) for o in rows)
    return {"orders": len(rows), "revenue": total, "customers": len(people),
            "per_customer": len(rows) / len(people),
            "per_order": total / len(rows)}


q1 = [o for o in ORDERS if o["quarter"] == "Q1"]
q2 = [o for o in ORDERS if o["quarter"] == "Q2"]
a, b = describe(q1), describe(q2)
print("Q1:", a)
print("Q2:", b)

Q1: {'orders': 114, 'revenue': 21000000, 'customers': 69, 'per_customer': 1.6521739130434783, 'per_order': 184210.52631578947}
Q2: {'orders': 86, 'revenue': 18700000, 'customers': 69, 'per_customer': 1.2463768115942029, 'per_order': 217441.86046511628}


## Rung 3: decompose along the tree

Revenue is customers, times how often each comes back, times what an order is worth.

In [10]:
rows = []
for name, key, fmt in (("customers", "customers", "{:,}"),
                       ("orders per customer", "per_customer", "{:.2f}"),
                       ("revenue per order", "per_order", "Rs {:,.0f}"),
                       ("revenue", "revenue", "Rs {:,.0f}")):
    change = 100 * (b[key] / a[key] - 1)
    rows.append((name, fmt.format(a[key]), fmt.format(b[key]), f"{change:+.1f}%"))
kit.table(["factor", "Q1", "Q2", "change"], rows, caption="Rung 3, the decomposition")

factor,Q1,Q2,change
customers,69,69,+0.0%
orders per customer,1.65,1.25,-24.6%
revenue per order,"Rs 184,211","Rs 217,442",+18.0%
revenue,"Rs 21,000,000","Rs 18,700,000",-11.0%


### The arithmetic has to close

If the three factors multiply to the revenue change, the decomposition is complete. If they do
not, a factor is missing or double counted and nothing below it can be trusted.

In [11]:
product = (b["customers"] / a["customers"]) * (b["per_customer"] / a["per_customer"]) \
    * (b["per_order"] / a["per_order"])
print(f"product of the three factor changes: {product:.3f}")
print(f"actual revenue change:               {b['revenue'] / a['revenue']:.3f}")
kit.check("the decomposition closes", abs(product - b["revenue"] / a["revenue"]) < 0.001,
          f"{product:.4f} against {b['revenue'] / a['revenue']:.4f}")

product of the three factor changes: 0.890
actual revenue change:               0.890


**The factor nobody expects.** Revenue per order went **up** 18 percent while revenue went down.
One factor moving the right way is hiding how far the other one fell, and reporting the fall
without saying so understates the frequency problem by about a third.

In [12]:
kit.flow(["customers flat", "frequency down 24.6%", "order value up 18%", "revenue down 11%"],
         lit=2, title="Two factors moved, and they moved against each other")

## Rung 4: isolate the segment

The same function, called once per segment. This is why it returns rather than prints.

In [13]:
segments = sorted({o["segment"] for o in ORDERS})
rows, moves = [], {}
for seg in segments:
    s1 = describe([o for o in q1 if o["segment"] == seg])
    s2 = describe([o for o in q2 if o["segment"] == seg])
    change = 100 * (s2["per_customer"] / s1["per_customer"] - 1)
    moves[seg] = change
    rows.append((seg, s1["orders"], s2["orders"], f"{s1['per_customer']:.2f}",
                 f"{s2['per_customer']:.2f}", f"{change:+.1f}%"))
kit.table(["segment", "Q1 orders", "Q2 orders", "Q1 per customer", "Q2 per customer", "change"],
          rows, caption="Rung 4, orders per customer by segment")

segment,Q1 orders,Q2 orders,Q1 per customer,Q2 per customer,change
Business,20,17,1.82,1.55,-15.0%
Retail-Core,38,36,1.12,1.06,-5.3%
Retail-Plus,51,26,2.32,1.18,-49.0%
Student,5,7,2.50,3.50,+40.0%


In [14]:
worst = min(moves, key=moves.get)
print(f"the fall is concentrated in {worst}, at {moves[worst]:+.1f} percent")
kit.check("Retail-Plus carries the fall", worst == "Retail-Plus", f"got {worst}")
kit.check("Retail-Core is close to flat", moves["Retail-Core"] > -10,
          f"{moves['Retail-Core']:+.1f} percent")
kit.check("one segment is worse than every other by a wide margin",
          moves[worst] < min(v for k, v in moves.items() if k != worst) - 20,
          f"{moves[worst]:+.1f} percent against the next worst")

the fall is concentrated in Retail-Plus, at -49.0 percent


## Rung 5: the hypothesis, and what would settle it

In [15]:
kit.vflow(["a complaint: the reorder button broke six weeks ago",
           "a hypothesis: that is why Retail-Plus frequency fell",
           "a test: reorder events per member, either side, against Core",
           "a finding, or a theory discarded, and both are progress"],
          lit=2, title="A cause is not a finding until a test is named beside it")

In [16]:
kit.check_summary()

## The sentence you would send both of them

> "Customers are flat at 69 in both quarters, so this is not acquisition. Orders per customer fell
> 24.6 percent, and almost all of that sits in Retail-Plus, which is down 49 percent against
> Retail-Core's 5. The reorder-feature complaint is a plausible cause and I have not tested it;
> reorder events per member either side of the six weeks, against Retail-Core, would settle it.
> One caveat: revenue per order rose 18 percent, which is masking about a third of the fall."

---

**What this file cannot tell you.** Nothing here measures the reorder feature, so the cause stays
a hypothesis. And every number above was computed on an ERP export nobody has profiled. Tomorrow
Finance says the export disagrees with their books, and some of today's answer moves.